In [2]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 24.3 MB/s  0:00:05 eta 0:00:010:00:01


In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import AdaBoostClassifier, VotingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
import warnings

warnings.filterwarnings('ignore')

# =============================================================================
# 1. CARGA DE DATOS ÓPTIMOS 
# =============================================================================
train_df = pd.read_csv('train_selected_RandomForest.csv')
tuning_df = pd.read_csv('tuning_selected_RandomForest.csv')

target_col = 'Class'
X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
X_tuning = tuning_df.drop(columns=[target_col])
y_tuning = tuning_df[target_col]

# --- ¡FIX DE XGBOOST! ---
X_train.columns = X_train.columns.str.replace(r'[\[\]<]', '_', regex=True)
X_tuning.columns = X_tuning.columns.str.replace(r'[\[\]<]', '_', regex=True)
# -----------------------------------

# Codificación del objetivo a formato numérico
le = LabelEncoder()
y_train_num = le.fit_transform(y_train)
y_tuning_num = le.transform(y_tuning)

# --- DEFINICIÓN DE VARIABLES PARA MÉTRICAS DINÁMICAS ---
# Esto debe ir ANTES de definir los modelos para que XGBoost sepa qué métrica usar
is_multiclass = len(le.classes_) > 2
avg_method = 'macro' if is_multiclass else 'binary'

# =============================================================================
# 2. DEFINICIÓN DE COMPONENTES PARA EL ENSAMBLE MANUAL
# =============================================================================
# Aquí están tus tres mejores modelos con los hiperparámetros ganadores congelados
rf_base = RandomForestClassifier(n_estimators=100, min_samples_split=2, min_samples_leaf=2, max_depth=None, criterion='entropy', random_state=42)
knn_base = KNeighborsClassifier(n_neighbors=5, p=1, weights='distance') 
svm_base = SVC(C=100, gamma=0.01, kernel='rbf', probability=True, random_state=42)

# =============================================================================
# 3. CONSTRUCCIÓN DE LOS MODELOS POR ENSEMBLE
# =============================================================================
modelos_ensemble = {
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    
    'XGBoost': XGBClassifier(
        n_estimators=100, 
        learning_rate=0.1, 
        eval_metric='mlogloss' if is_multiclass else 'logloss', 
        random_state=42
    ),
    
    'Ensamble Manual (Votación Soft)': VotingClassifier(
        estimators=[
            ('rf', rf_base),
            ('knn', knn_base),
            ('svm', svm_base)
        ],
        voting='soft' # Promedia las probabilidades de los 3 modelos estrella
    )
}

# =============================================================================
# 4. ENTRENAMIENTO Y EVALUACIÓN EN EL CONJUNTO DE TUNING
# =============================================================================
resultados_ensemble = []

print("Entrenando y evaluando modelos por Ensemble sobre el conjunto de Tuning...\n")

for nombre, modelo in modelos_ensemble.items():
    # Entrenamiento exclusivo con Train
    modelo.fit(X_train, y_train_num)
    
    # Predicciones en Tuning
    y_pred = modelo.predict(X_tuning)
    y_prob = modelo.predict_proba(X_tuning)
    
    # Métricas
    acc = accuracy_score(y_tuning_num, y_pred)
    prec = precision_score(y_tuning_num, y_pred, average=avg_method)
    rec = recall_score(y_tuning_num, y_pred, average=avg_method)
    f1 = f1_score(y_tuning_num, y_pred, average=avg_method)
    
    if is_multiclass:
        roc = roc_auc_score(y_tuning_num, y_prob, multi_class='ovo')
    else:
        roc = roc_auc_score(y_tuning_num, y_prob[:, 1])
        
    resultados_ensemble.append({
        'Modelo Ensemble': nombre,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': roc
    })

# =============================================================================
# 5. REPORTE FINAL DE RENDIMIENTO
# =============================================================================
df_ensemble_res = pd.DataFrame(resultados_ensemble).set_index('Modelo Ensemble')

print("="*85)
print("🏆 RENDIMIENTO DE LOS MODELOS POR ENSEMBLE CONSTRUIDOS (Tuning Set)")
print("="*85)
print(df_ensemble_res.round(4).to_string())
print("="*85)

Entrenando y evaluando modelos por Ensemble sobre el conjunto de Tuning...

🏆 RENDIMIENTO DE LOS MODELOS POR ENSEMBLE CONSTRUIDOS (Tuning Set)
                                 Accuracy  Precision  Recall  F1-Score  ROC-AUC
Modelo Ensemble                                                                
AdaBoost                           0.9194     0.9062  0.9355    0.9206   0.9292
XGBoost                            0.8387     0.8621  0.8065    0.8333   0.9126
Ensamble Manual (Votación Soft)    0.8871     0.9000  0.8710    0.8852   0.9157
